# 🛍️ Retail Promotions Data Analysis
### Codebasics Assignment — All 10 Questions

**Key Metrics:**
- **IR%** = ((Revenue After Promo - Revenue Before Promo) / Revenue Before Promo) × 100
- **ISU%** = ((Qty Sold After Promo - Qty Sold Before Promo) / Qty Sold Before Promo) × 100

## 📦 Setup & Data Loading

In [4]:
import os

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ─── Load all 4 datasets ───────────────────────────────────────────────────
# Gets the folder where the notebook is located
BASE_DIR = os.path.dirname(os.path.abspath('retail_promo_analysis.ipynb'))
DATA_DIR = os.path.join(BASE_DIR, 'datasets')

campaigns = pd.read_csv(os.path.join(DATA_DIR, 'dim_campaigns.csv'))
products  = pd.read_csv(os.path.join(DATA_DIR, 'dim_products.csv'))
stores    = pd.read_csv(os.path.join(DATA_DIR, 'dim_stores.csv'))
events    = pd.read_csv(os.path.join(DATA_DIR, 'fact_events.csv'))


print('✅ Datasets loaded successfully!')
print(f'   dim_campaigns : {campaigns.shape}')
print(f'   dim_products  : {products.shape}')
print(f'   dim_stores    : {stores.shape}')
print(f'   fact_events   : {events.shape}')

✅ Datasets loaded successfully!
   dim_campaigns : (2, 4)
   dim_products  : (15, 3)
   dim_stores    : (50, 2)
   fact_events   : (1510, 9)


In [5]:
# Quick preview of each table
print('=== dim_campaigns ===')
display(campaigns.head(3))

print('\n=== dim_products ===')
display(products.head(3))

print('\n=== dim_stores ===')
display(stores.head(3))

print('\n=== fact_events ===')
display(events.head(3))

=== dim_campaigns ===


,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023
1,CAMP_SAN_01,Sankranti,10-01-2024,16-01-2024



=== dim_products ===


,product_code,product_name,category
0,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples
1,P02,Atliq_Sonamasuri_Rice (10KG),Grocery & Staples
2,P03,Atliq_Suflower_Oil (1L),Grocery & Staples



=== dim_stores ===


,store_id,city
0,STTRV-0,Trivandrum
1,STMDU-3,Madurai
2,STHYD-6,Hyderabad



=== fact_events ===


,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488


---
## ❓ Q1 — Remove Duplicate Rows in Events Data
**Task:** Remove duplicates based on `store_id`, `campaign_id`, and `product_code`. How many duplicate rows were removed?

In [6]:
rows_before = len(events)

events_clean = events.drop_duplicates(subset=['store_id', 'campaign_id', 'product_code'])

rows_after   = len(events_clean)
duplicates_removed = rows_before - rows_after

print(f'Rows before deduplication : {rows_before}')
print(f'Rows after  deduplication : {rows_after}')
print(f'✅ Duplicate rows removed  : {duplicates_removed}')

Rows before deduplication : 1510
Rows after  deduplication : 1500
✅ Duplicate rows removed  : 10


---
## ❓ Q2 — Cities with More Than 5 Stores

In [7]:
city_store_count = stores.groupby('city')['store_id'].count().reset_index()
city_store_count.columns = ['city', 'store_count']

cities_more_than_5 = city_store_count[city_store_count['store_count'] > 5]

print(f'✅ Number of cities with more than 5 stores: {len(cities_more_than_5)}')
print()
display(cities_more_than_5.sort_values('store_count', ascending=False).reset_index(drop=True))

✅ Number of cities with more than 5 stores: 3



,city,store_count
0,Bengaluru,10
1,Chennai,8
2,Hyderabad,7


---
## ❓ Q3 — Fill Missing Values in `quantity_sold(before_promo)` Using Median
**Task:** Impute missing values using the median. How many were filled? What is the median?

In [8]:
col = 'quantity_sold(before_promo)'

missing_count = events_clean[col].isnull().sum()
median_value  = events_clean[col].median()

events_clean[col] = events_clean[col].fillna(median_value)

print(f'✅ Missing values filled : {missing_count}')
print(f'✅ Median used           : {median_value}')

✅ Missing values filled : 20
✅ Median used           : 78.0


---
## ❓ Q4 — Product Category with the Lowest Base Price (Before Promo)

In [9]:
# Merge events with products to get category info
events_prod = events_clean.merge(products, on='product_code', how='left')

avg_base_price_by_category = (
    events_prod
    .groupby('category')['base_price(before_promo)']
    .mean()
    .reset_index()
    .sort_values('base_price(before_promo)')
)

lowest_category = avg_base_price_by_category.iloc[0]

print(f'✅ Category with LOWEST avg base price: {lowest_category["category"]}')
print(f'   Average base price: ₹{lowest_category["base_price(before_promo)"]:.2f}')
print()
display(avg_base_price_by_category.reset_index(drop=True))

✅ Category with LOWEST avg base price: Personal Care
   Average base price: ₹102.38



,category,base_price(before_promo)
0,Personal Care,102.375
1,Grocery & Staples,385.000
2,Home Care,490.000
3,Home Appliances,685.000
4,Combo1,3000.000


---
## ❓ Q5 — Total Quantity Sold After Promo for BOGOF During Diwali Campaign

In [10]:
# Merge with campaigns
events_full = events_clean.merge(campaigns, on='campaign_id', how='left')

# BOGOF: customer pays for 1, gets 2 — so actual units = quantity_sold(after_promo) * 2
# Note: Some interpretations keep it as-is; adjust the multiplier below if needed
bogof_diwali = events_full[
    (events_full['campaign_name'].str.lower() == 'diwali') &
    (events_full['promo_type'].str.upper() == 'BOGOF')
]

total_qty_bogof_diwali = bogof_diwali['quantity_sold(after_promo)'].sum()

# For BOGOF, actual units dispensed = qty * 2
actual_units_bogof = total_qty_bogof_diwali * 2

print(f'✅ Total quantity_sold(after_promo) for BOGOF / Diwali: {total_qty_bogof_diwali}')
print(f'   (Actual units incl. free items @ BOGOF)           : {actual_units_bogof}')

✅ Total quantity_sold(after_promo) for BOGOF / Diwali: 34461
   (Actual units incl. free items @ BOGOF)           : 68922


---
## ❓ Q6 — Store with Highest Quantity Sold After Promo During Diwali

In [11]:
diwali_events = events_full[events_full['campaign_name'].str.lower() == 'diwali']

store_qty_diwali = (
    diwali_events
    .groupby('store_id')['quantity_sold(after_promo)']
    .sum()
    .reset_index()
    .sort_values('quantity_sold(after_promo)', ascending=False)
)

# Merge city info
store_qty_diwali = store_qty_diwali.merge(stores, on='store_id', how='left')

top_store = store_qty_diwali.iloc[0]

print(f'✅ Store with HIGHEST qty sold after promo (Diwali):')
print(f'   Store ID : {top_store["store_id"]}')
print(f'   City     : {top_store["city"]}')
print(f'   Qty Sold : {top_store["quantity_sold(after_promo)"]}')
print()
display(store_qty_diwali.head(5).reset_index(drop=True))

✅ Store with HIGHEST qty sold after promo (Diwali):
   Store ID : STCHE-4
   City     : Chennai
   Qty Sold : 5013



,store_id,quantity_sold(after_promo),city
0,STCHE-4,5013,Chennai
1,STBLR-7,4893,Bengaluru
2,STBLR-6,4857,Bengaluru
3,STMYS-1,4779,Mysuru
4,STCHE-7,4779,Chennai


---
## ❓ Q7 — Compare Sankranti vs Diwali: Which Campaign Saw Greater Sales Increase?

In [12]:
campaign_comparison = (
    events_full[
        events_full['campaign_name'].str.lower().isin(['diwali', 'sankranti'])
    ]
    .groupby('campaign_name')
    .agg(
        total_qty_before=('quantity_sold(before_promo)', 'sum'),
        total_qty_after =('quantity_sold(after_promo)',  'sum')
    )
    .reset_index()
)

campaign_comparison['increase']   = campaign_comparison['total_qty_after'] - campaign_comparison['total_qty_before']
campaign_comparison['increase_%'] = ((campaign_comparison['increase'] / campaign_comparison['total_qty_before']) * 100).round(2)

display(campaign_comparison)

winner = campaign_comparison.loc[campaign_comparison['increase'].idxmax(), 'campaign_name']
print(f'\n✅ Campaign with GREATER increase in sales: {winner}')

,campaign_name,total_qty_before,total_qty_after,increase,increase_%
0,Diwali,109756.0,183404,73648.0,67.10
1,Sankranti,97894.0,252069,154175.0,157.49



✅ Campaign with GREATER increase in sales: Sankranti


---
## ❓ Q8 — Product with Highest IR% During Sankranti Campaign

**Formula:**
```
Revenue Before = base_price(before_promo) × quantity_sold(before_promo)
Revenue After  = base_price(after_promo)  × quantity_sold(after_promo)
IR% = ((Revenue After - Revenue Before) / Revenue Before) × 100
```

In [13]:
sankranti = events_full[
    events_full['campaign_name'].str.lower() == 'sankranti'
].copy()

sankranti['revenue_before'] = sankranti['base_price(before_promo)'] * sankranti['quantity_sold(before_promo)']
sankranti['revenue_after']  = sankranti['base_price(after_promo)']  * sankranti['quantity_sold(after_promo)']

product_ir = (
    sankranti
    .groupby('product_code')
    .agg(
        rev_before=('revenue_before', 'sum'),
        rev_after =('revenue_after',  'sum')
    )
    .reset_index()
)

product_ir['IR%'] = ((product_ir['rev_after'] - product_ir['rev_before']) / product_ir['rev_before'] * 100).round(2)

# Merge product names
product_ir = product_ir.merge(products[['product_code', 'product_name', 'category']], on='product_code', how='left')
product_ir = product_ir.sort_values('IR%', ascending=False)

top_product = product_ir.iloc[0]

print(f'✅ Product with HIGHEST IR% during Sankranti:')
print(f'   Product  : {top_product["product_name"]}')
print(f'   Category : {top_product["category"]}')
print(f'   IR%      : {top_product["IR%"]}%')
print()
display(product_ir.head(5).reset_index(drop=True))

✅ Product with HIGHEST IR% during Sankranti:
   Product  : Atliq_Suflower_Oil (1L)
   Category : Grocery & Staples
   IR%      : 91.83%



,product_code,rev_before,rev_after,IR%,product_name,category
0,P03,3189600.0,6118500,91.83,Atliq_Suflower_Oil (1L),Grocery & Staples
1,P15,16185000.0,31027500,91.71,Atliq_Home_Essential_8_Product_Combo,Combo1
2,P13,1740550.0,3303125,89.77,Atliq_High_Glo_15W_LED_Bulb,Home Appliances
3,P14,4542060.0,8534850,87.91,Atliq_waterproof_Immersion_Rod,Home Appliances
4,P04,6813550.0,12779800,87.56,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples


---
## ❓ Q9 — Store in Visakhapatnam with Lowest ISU% During Diwali

**Formula:**
```
ISU% = ((quantity_sold(after_promo) - quantity_sold(before_promo)) / quantity_sold(before_promo)) × 100
```

In [14]:
# Merge stores into events_full
events_full_stores = events_full.merge(stores, on='store_id', how='left')

vizag_diwali = events_full_stores[
    (events_full_stores['campaign_name'].str.lower() == 'diwali') &
    (events_full_stores['city'].str.lower() == 'visakhapatnam')
].copy()

store_isu = (
    vizag_diwali
    .groupby('store_id')
    .agg(
        qty_before=('quantity_sold(before_promo)', 'sum'),
        qty_after =('quantity_sold(after_promo)',  'sum')
    )
    .reset_index()
)

store_isu['ISU%'] = ((store_isu['qty_after'] - store_isu['qty_before']) / store_isu['qty_before'] * 100).round(2)
store_isu = store_isu.sort_values('ISU%')

lowest_store = store_isu.iloc[0]

print(f'✅ Store in Visakhapatnam with LOWEST ISU% (Diwali):')
print(f'   Store ID : {lowest_store["store_id"]}')
print(f'   ISU%     : {lowest_store["ISU%"]}%')
print()
display(store_isu.reset_index(drop=True))

✅ Store in Visakhapatnam with LOWEST ISU% (Diwali):
   Store ID : STVSK-3
   ISU%     : 49.21%



,store_id,qty_before,qty_after,ISU%
0,STVSK-3,1780.0,2656,49.21
1,STVSK-4,1926.0,2908,50.99
2,STVSK-1,1903.0,3078,61.74
3,STVSK-2,1701.0,2860,68.14
4,STVSK-0,1768.0,3005,69.97


---
## ❓ Q10 — Promo Type with BOTH Negative IR% AND Negative ISU% During Sankranti

In [15]:
sankranti2 = events_full[
    events_full['campaign_name'].str.lower() == 'sankranti'
].copy()

sankranti2['revenue_before'] = sankranti2['base_price(before_promo)'] * sankranti2['quantity_sold(before_promo)']
sankranti2['revenue_after']  = sankranti2['base_price(after_promo)']  * sankranti2['quantity_sold(after_promo)']

promo_metrics = (
    sankranti2
    .groupby('promo_type')
    .agg(
        qty_before =('quantity_sold(before_promo)', 'sum'),
        qty_after  =('quantity_sold(after_promo)',  'sum'),
        rev_before =('revenue_before',              'sum'),
        rev_after  =('revenue_after',               'sum')
    )
    .reset_index()
)

promo_metrics['IR%']  = ((promo_metrics['rev_after']  - promo_metrics['rev_before'])  / promo_metrics['rev_before']  * 100).round(2)
promo_metrics['ISU%'] = ((promo_metrics['qty_after']  - promo_metrics['qty_before'])  / promo_metrics['qty_before']  * 100).round(2)

display(promo_metrics[['promo_type', 'IR%', 'ISU%']])

negative_both = promo_metrics[
    (promo_metrics['IR%'] < 0) & (promo_metrics['ISU%'] < 0)
]

print(f'\n✅ Promo type(s) with BOTH negative IR% and ISU% (Sankranti):')
if len(negative_both) > 0:
    for _, row in negative_both.iterrows():
        print(f'   {row["promo_type"]} → IR%: {row["IR%"]}%  |  ISU%: {row["ISU%"]}%')
else:
    print('   None found.')

,promo_type,IR%,ISU%
0,25% OFF,-39.33,-19.60
1,33% OFF,-5.90,41.15
2,50% OFF,-31.08,37.05
3,500 Cashback,91.71,130.05
4,BOGOF,88.33,278.04



✅ Promo type(s) with BOTH negative IR% and ISU% (Sankranti):
   25% OFF → IR%: -39.33%  |  ISU%: -19.6%


---
## 📊 Summary of All Answers

In [16]:
print('=' * 60)
print('          SUMMARY OF ALL ANSWERS')
print('=' * 60)
print(f'Q1  Duplicate rows removed          : {duplicates_removed}')
print(f'Q2  Cities with > 5 stores          : {len(cities_more_than_5)}')
print(f'Q3  Missing values filled           : {missing_count}  |  Median: {median_value}')
print(f'Q4  Category with lowest base price : {lowest_category["category"]}')
print(f'Q5  BOGOF Diwali qty (after promo)  : {total_qty_bogof_diwali}')
print(f'Q6  Top store (Diwali)              : {top_store["store_id"]} — {top_store["city"]}')
print(f'Q7  Campaign with greater increase  : {winner}')
print(f'Q8  Highest IR% product (Sankranti) : {top_product["product_name"]} ({top_product["IR%"]}%)')
print(f'Q9  Lowest ISU% store in Vizag      : {lowest_store["store_id"]} ({lowest_store["ISU%"]}%)')
print(f'Q10 Promo with -ve IR% & -ve ISU%   : See output above')
print('=' * 60)

          SUMMARY OF ALL ANSWERS
Q1  Duplicate rows removed          : 10
Q2  Cities with > 5 stores          : 3
Q3  Missing values filled           : 20  |  Median: 78.0
Q4  Category with lowest base price : Personal Care
Q5  BOGOF Diwali qty (after promo)  : 34461
Q6  Top store (Diwali)              : STCHE-4 — Chennai
Q7  Campaign with greater increase  : Sankranti
Q8  Highest IR% product (Sankranti) : Atliq_Suflower_Oil (1L) (91.83%)
Q9  Lowest ISU% store in Vizag      : STVSK-3 (49.21%)
Q10 Promo with -ve IR% & -ve ISU%   : See output above
